# 图像编码 · 学生版 Notebook

只保留课堂可操作观察。答案检查单元由教师决定何时开放。

## 路径检查

In [ ]:
from pathlib import Path
from PIL import Image, ImageDraw, ImageFilter
from IPython.display import display
import struct, os

lesson_dir = Path.cwd()
assets_dir = lesson_dir / "assets"
if not assets_dir.is_dir() and lesson_dir.name == "notebooks" and (lesson_dir.parent / "assets").is_dir():
    os.chdir(lesson_dir.parent)
    lesson_dir = Path.cwd()
    assets_dir = lesson_dir / "assets"
assert assets_dir.is_dir(), f"请从本课目录启动 notebook。当前目录：{Path.cwd()}"
print("lesson directory:", lesson_dir)
print("assets directory:", assets_dir)

## Q1 / Q3 · 生成固定测试图

In [ ]:
# Q1/Q3 共用固定测试图：无随机数，每次结果一致
path = Path("assets/test-image.png")
path.parent.mkdir(parents=True, exist_ok=True)
w = h = 256
img = Image.new("RGB", (w, h))
px = img.load()
for y in range(h):
    for x in range(w):
        v = round(255 * x / (w - 1))
        px[x, y] = (v, v, v)
draw = ImageDraw.Draw(img)
draw.rectangle((12, 12, 72, 72), fill=(255, 0, 0))
draw.rectangle((73, 12, 133, 72), fill=(0, 255, 0))
draw.rectangle((184, 12, 244, 72), fill=(0, 0, 255))
draw.rectangle((0, 104, 128, 255), fill=(224, 224, 224))
draw.rectangle((136, 104, 255, 255), fill=(224, 224, 224))
draw.ellipse((0, 112, 128, 240), outline=(0, 0, 0), width=28)
draw.line((144, 240, 240, 144), fill=(0, 0, 0), width=28)
for x in [152, 160, 168, 176, 184, 192]:
    draw.line((x, 224, x, 244), fill=(255, 255, 255), width=2)
img.save(path)
assert img.size == (256, 256) and img.mode == "RGB"
print("created:", path)
display(img)

## Q1 · 尝试新的采样尺寸

In [ ]:
source_path = Path("assets/test-image.png")
src = Image.open(source_path).convert("RGB")
def show_sampling(n: int):
    sampled = src.resize((n, n), resample=Image.Resampling.NEAREST)
    enlarged = sampled.resize((512, 512), resample=Image.Resampling.NEAREST)
    print(f"{n} × {n}")
    display(enlarged)
    return enlarged
show_sampling(24)

## Q3 · 观察颜色数量减少

In [ ]:
# Q3 · Demo B：主课只看 original / 16-color / 2-color；先看图，后揭示接缝数值
source_path = Path("assets/test-image.png")
assert source_path.is_file()
src = Image.open(source_path).convert("RGB")
targets = [256, 16, 4, 2]
main_targets = [16, 2]
outputs = {}
for n in targets:
    q = src.quantize(colors=n, method=Image.Quantize.MEDIANCUT, dither=Image.Dither.NONE).convert("RGB")
    colors = q.getcolors(maxcolors=n + 1)
    assert colors is not None
    color_count = len(colors)
    assert q.size == src.size
    assert q.mode == "RGB"
    assert color_count <= n
    outputs[n] = q
print("original")
display(src)
for n in main_targets:
    print(f"max {n} colors")
    display(outputs[n])

## Q3 · 接缝证据（教师允许后运行）

In [ ]:
# Q3 · 接缝证据：学生观察解释之后再运行/揭示
seam_left = (72, 40)
seam_right = (73, 40)
for n in [16, 2]:
    left_rgb = outputs[n].getpixel(seam_left)
    right_rgb = outputs[n].getpixel(seam_right)
    print(f"{n:2d} colors | left={left_rgb} | right={right_rgb} | same={left_rgb == right_rgb}")

## Q6 · RGB → bits

In [ ]:
# Q6 · Demo C：RGB → bits
rgb = (255, 128, 0)
for name, value in zip(["R", "G", "B"], rgb):
    print(f"{name} {value:3d} → {value:08b}")
print("combined:", " ".join(f"{v:08b}" for v in rgb))